# Structured Contrastive Embedding — Distinct Semantic Subspaces

Uses `ts_embed.loss.StructuredContrastiveLoss` to make each embedding chunk
capture a **distinct** semantic. Four semantic groups:

| chunk | dims | semantic |
|---|---|---|
| `balance` | `[0:48]` | balance level / volatility / trend |
| `payment` | `[48:96]` | payment ratio / regularity |
| `delinquency` | `[96:144]` | delinquency frequency / severity |
| `other` | `[144:192]` | other account events |

Two forces are combined in the loss:

1. **Per-semantic SupCon** — each chunk has a private head + supervised-
   contrastive loss with semantic-specific positives, so it *specializes*.
2. **Cross-chunk decorrelation** — z-scored chunk activations are pushed to
   zero cross-correlation between *different* chunks, so they encode *distinct*
   (non-redundant) information. A per-chunk variance hinge stops decorrelation
   from being satisfied by collapsing dimensions.

Single-GPU here; the loss is DDP-safe and drops into `scripts/train_ddp.py`'s
launch pattern unchanged.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from ts_embed.data import (DatasetPaths, TimeSeriesDataset,
                           TimeFeatureMasker, ContrastiveCollator)
from ts_embed.model import TSEmbeddingModel, TSEncoderConfig
from ts_embed.loss import StructuredContrastiveLoss, SemanticSpec

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Synthetic data — four *independent* latent semantics

Feature blocks (F_NUM=98): balance `0:25`, payment `25:50`, delinquency
`50:75`, other `75:98`. Each is driven by its **own independent** latent
group. Independence is the test: a well-structured embedding's `balance`
chunk should recover only the balance group, etc.

In [ ]:
N, T, F_NUM, F_CAT = 6000, 24, 98, 2
BLK = {'balance': slice(0, 25), 'payment': slice(25, 50),
       'delinquency': slice(50, 75), 'other': slice(75, 98)}
K = 3  # latent groups per semantic

groups = {nm: np.random.randint(0, K, N) for nm in BLK}
numeric = 0.3 * np.random.randn(N, T, F_NUM).astype(np.float32)
t_axis = np.linspace(0, 1, T).astype(np.float32)

# balance: group sets level + linear trend slope
lvl = np.array([-1.0, 0.0, 1.2], np.float32); slope = np.array([0.0, 0.8, -0.6], np.float32)
for g in range(K):
    m = groups['balance'] == g
    curve = (lvl[g] + slope[g] * t_axis)[None, :, None]
    numeric[m, :, BLK['balance']] += curve

# payment: group sets payment ratio + regularity (temporal noise)
pr = np.array([0.2, 0.6, 0.95], np.float32); reg = np.array([0.5, 0.2, 0.05], np.float32)
for g in range(K):
    m = groups['payment'] == g
    numeric[m, :, BLK['payment']] += pr[g] + reg[g] * np.random.randn(
        m.sum(), T, 25).astype(np.float32)

# delinquency: group sets event frequency bursts
freq = np.array([0.02, 0.15, 0.45], np.float32)
for g in range(K):
    m = groups['delinquency'] == g
    bursts = (np.random.rand(m.sum(), T, 25) < freq[g]).astype(np.float32) * 2.0
    numeric[m, :, BLK['delinquency']] += bursts

# other: group sets a distinct templated pattern
other_tpl = np.random.randn(K, 23).astype(np.float32) * 1.3
for g in range(K):
    m = groups['other'] == g
    numeric[m, :, BLK['other']] += other_tpl[g][None, None, :]

missing = (np.random.rand(N, T, F_NUM) < 0.1).astype(np.uint8)
feat_mean = numeric.mean((0, 1), keepdims=True)
numeric = np.where(missing == 1, feat_mean, numeric).astype(np.float32)

# cat 0 = delinquency flag (tied to delinquency group); cat 1 = other-event flag
del_p = np.array([0.05, 0.4, 0.85], np.float32)[groups['delinquency']]
oth_p = np.array([0.1, 0.5, 0.9], np.float32)[groups['other']]
categorical = np.stack([
    (np.random.rand(N, T) < del_p[:, None]).astype(np.int8),
    (np.random.rand(N, T) < oth_p[:, None]).astype(np.int8),
], axis=-1)

data_dir = REPO_ROOT / 'data_demo'; data_dir.mkdir(exist_ok=True)
np.save(data_dir / 'numeric.npy', numeric)
np.save(data_dir / 'missing.npy', missing)
np.save(data_dir / 'categorical.npy', categorical)
print('saved', {p.name: np.load(p, mmap_mode='r').shape for p in sorted(data_dir.glob('*.npy'))})

## 2. Per-semantic descriptors -> positive labels

Each semantic gets a **time-series descriptor**: level (temporal mean),
volatility (temporal std), and a least-squares **trend slope** over the 24
months. Endpoint differences (`x[-1] - x[0]`) are too noisy for real series,
so we fit a proper slope. KMeans on each descriptor yields the cluster id used
as that semantic's positive label. Swap these for your production descriptors
(MCC mix, paydown ratios, delinquency counts, ...).

In [ ]:
def kmeans(X, k, seed=0):
    try:
        from sklearn.cluster import KMeans
        return KMeans(k, n_init=10, random_state=seed).fit_predict(X)
    except ImportError:
        rng = np.random.default_rng(seed)
        C = X[rng.choice(len(X), k, replace=False)]
        for _ in range(50):
            lab = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)
            C = np.stack([X[lab == j].mean(0) if (lab == j).any() else C[j]
                          for j in range(k)])
        return lab

raw = np.load(data_dir / 'numeric.npy'); cat = np.load(data_dir / 'categorical.npy')

# Centered month axis, reused for the least-squares trend slope.
t_idx = np.arange(T, dtype=np.float32)
t_ctr = t_idx - t_idx.mean()

def desc(block):
    b = raw[:, :, block]                                  # (N, T, fb)
    level = b.mean((1, 2))[:, None]                       # average level
    vol = b.std(1).mean(1)[:, None]                       # temporal volatility
    # OLS slope per feature = sum((x-xbar)*t_ctr) / sum(t_ctr^2); sum(t_ctr)=0.
    slope = (b * t_ctr[None, :, None]).sum(1) / (t_ctr ** 2).sum()
    trend = slope.mean(1)[:, None]                        # avg trend over feats
    return np.concatenate([level, vol, trend], 1)

balance_desc = desc(BLK['balance'])
payment_desc = desc(BLK['payment'])
deliq_desc = np.concatenate([desc(BLK['delinquency']), cat[:, :, 0].mean(1)[:, None]], 1)
other_desc = np.concatenate([desc(BLK['other']), cat[:, :, 1].mean(1)[:, None]], 1)

LABELS_NP = {
    'balance': kmeans(balance_desc, K).astype(np.int64),
    'payment': kmeans(payment_desc, K).astype(np.int64),
    'delinquency': kmeans(deliq_desc, K).astype(np.int64),
    'other': kmeans(other_desc, K).astype(np.int64),
}

def recovery(pred, true):
    from itertools import permutations
    return max(np.mean([p[t] == pr for t, pr in zip(true, pred)])
               for p in permutations(range(true.max() + 1)))
for nm in BLK:
    print(f'{nm:12s} cluster recovers its latent group: '
          f'{recovery(LABELS_NP[nm], groups[nm]):.2f}')

## 3. Dataset + label-carrying collator + train / val split

`TimeFeatureMasker` builds the masked view with **time-series-aware**
augmentation: contiguous month spans are hidden (not scattered single months,
which a sequence model could trivially copy from a neighbour) and each masked
feature channel loses a contiguous time window. The collator attaches the
per-semantic positive labels for each batch.

We hold out **15%** of accounts for **validation**. The labels were already
computed from descriptors (no target leakage), so we can split at random.
Best-model selection in the training loop is based on validation loss.

In [ ]:
paths = DatasetPaths(numeric=data_dir / 'numeric.npy',
                     missing=data_dir / 'missing.npy',
                     categorical=data_dir / 'categorical.npy')
base_ds = TimeSeriesDataset(paths)

class IndexedDataset(Dataset):
    def __init__(self, ds): self.ds = ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]; item['_idx'] = i; return item

LABELS = {k: torch.from_numpy(v) for k, v in LABELS_NP.items()}

class SemanticCollator:
    def __init__(self, masker):
        self.cc = ContrastiveCollator(masker)
    def __call__(self, batch):
        idx = torch.tensor([b.pop('_idx') for b in batch])
        out = self.cc(batch)
        out['labels'] = {k: v[idx] for k, v in LABELS.items()}
        return out

# 85 / 15 train / val split. random_split on IndexedDataset preserves the
# global index via `_idx` so the collator looks up the correct semantic
# labels regardless of which split a sample came from.
indexed = IndexedDataset(base_ds)
val_n = int(0.15 * len(indexed))
train_n = len(indexed) - val_n
train_ds, val_ds = torch.utils.data.random_split(
    indexed, [train_n, val_n], generator=torch.Generator().manual_seed(0))

masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30,
                           n_time_spans=2, feat_span_frac=0.5)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,
                          drop_last=True, num_workers=0,
                          collate_fn=SemanticCollator(masker))
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False,
                        drop_last=False, num_workers=0,
                        collate_fn=SemanticCollator(masker))
print(f'train batches/epoch: {len(train_loader)}  |  val batches/epoch: {len(val_loader)}')

## 4. Model + structured loss (4 chunks of 48 dims)

`d_model = 192` split into four equal 48-dim chunks. `decorr_weight` controls
how hard distinct chunks are pushed apart; `var_weight` keeps dims active.

In [ ]:
D_MODEL = 192
cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(2, 2), seq_len=T,
                      d_model=D_MODEL, n_layers=2, n_heads=4, proj_dim=128)
model = TSEmbeddingModel(cfg).to(device)

semantics = [
    SemanticSpec('balance',       0,  48, temperature=0.1, contrastive_weight=1.0),
    SemanticSpec('payment',      48,  96, temperature=0.1, contrastive_weight=1.0),
    SemanticSpec('delinquency',  96, 144, temperature=0.1, contrastive_weight=1.0),
    SemanticSpec('other',       144, 192, temperature=0.1, contrastive_weight=1.0),
]
struct_loss = StructuredContrastiveLoss(
    semantics, decorr_weight=1.0, var_weight=1.0, var_gamma=1.0
).to(device)

params = list(model.parameters()) + list(struct_loss.parameters())
optim = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
print('trainable params:', round(sum(p.numel() for p in params) / 1e6, 2), 'M')

## 5. Training loop with validation + best-model selection

Each epoch:

1. Train through `train_loader`; track running averages of every loss
   component.
2. Run a no-grad pass through `val_loader` with the masking RNG reseeded so
   the val loss is directly comparable across epochs.
3. If `val_loss` improved, snapshot the model + structured-loss state into
   `best_state`.

At the end, save the best checkpoint to `runs/structured/best.pt` and reload
it for evaluation. Watch for:

- **`c_*`** falling on both train and val -- the per-semantic SupCon terms
  are learning.
- **`decorr`** falling -- cross-chunk correlation shrinking (distinct
  subspaces).
- **`val_loss` diverging upward** while `train_loss` keeps falling -- the
  encoder is overfitting the cluster labels.

In [ ]:
EPOCHS = 12
history = []
best_val_loss = float('inf')
best_state = None

@torch.no_grad()
def run_val():
    """Mean structured-loss components on the validation split. Reseeds the
    RNG so the random masking is identical across epochs."""
    model.eval(); struct_loss.eval()
    rng_state = torch.get_rng_state(); torch.manual_seed(123)
    sums = {}; nb_local = 0
    try:
        for batch in val_loader:
            num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
            keep_a = batch['time_keep_a'].to(device)
            num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
            keep_b = batch['time_keep_b'].to(device)
            cat_a = batch['categorical_a'].to(device)
            cat_b = batch['categorical_b'].to(device)
            labels = {k: v.to(device) for k, v in batch['labels'].items()}
            emb_a = model.encode(num_a, mis_a, cat_a, keep_a)
            emb_b = model.encode(num_b, mis_b, cat_b, keep_b)
            out = struct_loss(emb_a, emb_b, labels=labels)
            for k, v in out.items():
                sums[k] = sums.get(k, 0.0) + float(v)
            nb_local += 1
    finally:
        torch.set_rng_state(rng_state)
    return {k: v / max(nb_local, 1) for k, v in sums.items()}


for epoch in range(EPOCHS):
    model.train(); struct_loss.train()
    train_sums = {}; nb_local = 0
    for batch in train_loader:
        num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
        keep_a = batch['time_keep_a'].to(device)
        num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
        keep_b = batch['time_keep_b'].to(device)
        cat_a = batch['categorical_a'].to(device)
        cat_b = batch['categorical_b'].to(device)
        labels = {k: v.to(device) for k, v in batch['labels'].items()}

        emb_a = model.encode(num_a, mis_a, cat_a, keep_a)
        emb_b = model.encode(num_b, mis_b, cat_b, keep_b)
        out = struct_loss(emb_a, emb_b, labels=labels)

        optim.zero_grad(set_to_none=True)
        out['loss'].backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optim.step()
        for k, v in out.items():
            train_sums[k] = train_sums.get(k, 0.0) + float(v)
        nb_local += 1

    train_avg = {k: v / max(nb_local, 1) for k, v in train_sums.items()}
    val_avg = run_val()
    history.append({'epoch': epoch, 'train': train_avg, 'val': val_avg})

    improved = val_avg['loss'] < best_val_loss
    if improved:
        best_val_loss = val_avg['loss']
        best_state = {
            'model':       {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            'struct_loss': {k: v.detach().cpu().clone() for k, v in struct_loss.state_dict().items()},
            'cfg': cfg.__dict__,
            'epoch': epoch,
            'val_loss': val_avg['loss'],
            'val_components': {k: v for k, v in val_avg.items() if k != 'loss'},
        }

    print(f"epoch {epoch:2d}  train={train_avg['loss']:.3f}  val={val_avg['loss']:.3f}  "
          f"(val: c_bal={val_avg['c_balance']:.3f} c_pay={val_avg['c_payment']:.3f} "
          f"c_del={val_avg['c_delinquency']:.3f} c_oth={val_avg['c_other']:.3f}  "
          f"decorr={val_avg['decorr']:.4f})"
          + ('  <- best' if improved else ''))

run_dir = REPO_ROOT / 'runs' / 'structured'
run_dir.mkdir(parents=True, exist_ok=True)
torch.save(best_state, run_dir / 'best.pt')
print(f'\nbest val_loss = {best_val_loss:.3f}  @ epoch {best_state["epoch"]}  ->  {run_dir / "best.pt"}')

## 6. Reload best model + specialization + disentanglement checks

Reload the best-by-`val_loss` checkpoint, extract embeddings for every
account, and run two diagnostics:

1. **Specialization** -- kNN-classify every (chunk, semantic-label) pair. The
   4 x 4 accuracy matrix should be **high on the diagonal**, near chance off
   it.
2. **Disentanglement** -- mean |cross-correlation| between chunks should be
   **near zero off-diagonal**.

Plus a per-chunk std collapse guard.

In [ ]:
# Reload the best-validated checkpoint into the live model.
model.load_state_dict(best_state['model'])
struct_loss.load_state_dict(best_state['struct_loss'])

model.eval()
embs = []
with torch.inference_mode():
    for i in range(0, N, 512):
        sl = slice(i, min(i + 512, N))
        n = torch.from_numpy(np.load(data_dir/'numeric.npy', mmap_mode='r')[sl].copy()).to(device)
        m = torch.from_numpy(np.load(data_dir/'missing.npy', mmap_mode='r')[sl].copy()).float().to(device)
        c = torch.from_numpy(np.load(data_dir/'categorical.npy', mmap_mode='r')[sl].astype(np.int64)).to(device)
        embs.append(model.encode(n, m, c).float().cpu().numpy())
embs = np.concatenate(embs)

CH = {'balance': slice(0, 48), 'payment': slice(48, 96),
      'delinquency': slice(96, 144), 'other': slice(144, 192)}

def knn_acc(X, y, k=15, seed=0):
    rng_local = np.random.default_rng(seed)
    perm = rng_local.permutation(len(X)); tr, te = perm[:4500], perm[4500:]
    Xt = X[tr] / (np.linalg.norm(X[tr], axis=1, keepdims=True) + 1e-8)
    Xe = X[te] / (np.linalg.norm(X[te], axis=1, keepdims=True) + 1e-8)
    nn = np.argpartition(-(Xe @ Xt.T), k, axis=1)[:, :k]
    pred = np.array([np.bincount(r).argmax() for r in y[tr][nn]])
    return float((pred == y[te]).mean())

names = list(CH)
print('kNN accuracy  (rows=chunk, cols=semantic label)  chance=%.2f' % (1.0 / K))
print('              ' + '  '.join(f'{n:>11s}' for n in names))
knn_mat = np.zeros((len(names), len(names)))
for ci, cn in enumerate(names):
    for li, ln in enumerate(names):
        knn_mat[ci, li] = knn_acc(embs[:, CH[cn]], LABELS_NP[ln])
    print(f'{cn:12s}  ' + '  '.join(f'{v:11.3f}' for v in knn_mat[ci]))

# Disentanglement: mean |Pearson r| between chunk activations.
Z = {cn: (embs[:, CH[cn]] - embs[:, CH[cn]].mean(0)) /
         (embs[:, CH[cn]].std(0) + 1e-8) for cn in names}
print('\nmean |cross-correlation| between chunks:')
print('              ' + '  '.join(f'{n:>11s}' for n in names))
corr_mat = np.zeros((len(names), len(names)))
for ai, a in enumerate(names):
    for bi, b in enumerate(names):
        corr_mat[ai, bi] = float(np.abs(Z[a].T @ Z[b] / len(embs)).mean())
    print(f'{a:12s}  ' + '  '.join(f'{v:11.3f}' for v in corr_mat[ai]))

for cn in names:
    s = embs[:, CH[cn]].std(0)
    assert s.mean() > 1e-2, f'COLLAPSE in {cn} chunk'
print('\nExpect: kNN diagonal high / off-diagonal ~chance, and cross-correlation'
      ' off-diagonal ~0 => distinct semantic subspaces.')

## 7. Visualization

Three plots:

1. **Training curves** -- train vs val loss across epochs, with the
   best-val epoch annotated.
2. **kNN-accuracy and cross-correlation heatmaps** -- summary view of
   specialization (diagonal-heavy) and disentanglement (off-diagonal ~0).
3. **Per-chunk PCA scatter grid** -- 4 x 4 panel. Each row is one chunk
   projected to 2D via PCA; each column colors the points by one of the four
   semantic labels. The **diagonal** (green border) should show clear
   clusters (chunk specialized for its own aspect); **off-diagonal** should
   look diffuse (the chunk doesn't encode the other aspect).

In [ ]:
try:
    import matplotlib.pyplot as plt

    epochs = [h['epoch'] for h in history]
    tr = [h['train']['loss'] for h in history]
    va = [h['val']['loss']   for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(epochs, tr, '-o', label='train', linewidth=2)
    axes[0].plot(epochs, va, '-s', label='val',   linewidth=2)
    axes[0].axvline(best_state['epoch'], color='k', linestyle=':', alpha=0.6,
                     label=f"best epoch {best_state['epoch']}")
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('total loss')
    axes[0].set_title('train / val loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    im1 = axes[1].imshow(knn_mat, vmin=0, vmax=1, cmap='viridis')
    axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels(names, rotation=30, ha='right')
    axes[1].set_yticks(range(len(names))); axes[1].set_yticklabels(names)
    axes[1].set_xlabel('label aspect'); axes[1].set_ylabel('chunk')
    axes[1].set_title(f'kNN accuracy  (chance={1.0/K:.2f})')
    for i in range(len(names)):
        for j in range(len(names)):
            axes[1].text(j, i, f'{knn_mat[i, j]:.2f}', ha='center', va='center',
                         color='white' if knn_mat[i, j] < 0.55 else 'black', fontsize=9)
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(corr_mat, vmin=0, vmax=max(corr_mat.max(), 0.05),
                         cmap='magma')
    axes[2].set_xticks(range(len(names))); axes[2].set_xticklabels(names, rotation=30, ha='right')
    axes[2].set_yticks(range(len(names))); axes[2].set_yticklabels(names)
    axes[2].set_title('mean |cross-correlation| between chunks')
    for i in range(len(names)):
        for j in range(len(names)):
            axes[2].text(j, i, f'{corr_mat[i, j]:.2f}', ha='center', va='center',
                         color='white' if corr_mat[i, j] < 0.5 * corr_mat.max() else 'black', fontsize=9)
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed; skipping training-curve and heatmap plots')

In [ ]:
try:
    import matplotlib.pyplot as plt

    def pca2d(X, seed=0):
        try:
            from sklearn.decomposition import PCA
            return PCA(n_components=2, random_state=seed).fit_transform(X)
        except ImportError:
            Xc = X - X.mean(0)
            _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
            return Xc @ Vt[:2].T

    # Subsample for cleaner plots.
    rng_local = np.random.default_rng(0)
    n_show = min(2500, len(embs))
    sample_idx = rng_local.choice(len(embs), size=n_show, replace=False)

    n = len(names)
    fig, axes = plt.subplots(n, n, figsize=(14, 14))
    for i, chunk_name in enumerate(names):
        Z2 = pca2d(embs[sample_idx][:, CH[chunk_name]])
        for j, label_name in enumerate(names):
            ax = axes[i, j]
            colors = LABELS_NP[label_name][sample_idx]
            ax.scatter(Z2[:, 0], Z2[:, 1], c=colors, cmap='tab10', s=6, alpha=0.65)
            if i == 0:
                ax.set_title(f'colored by\n{label_name}', fontsize=10)
            if j == 0:
                ax.set_ylabel(f'{chunk_name} chunk\n(PCA 2D)', fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])
            if i == j:
                for spine in ax.spines.values():
                    spine.set_edgecolor('green'); spine.set_linewidth(2.5)
    fig.suptitle("Each chunk's 2D PCA, colored by every semantic label.  "
                 "Diagonal (green border) = chunk vs its OWN label -- should show CLEAR clusters.  "
                 "Off-diagonal = chunk vs OTHER aspect -- should look DIFFUSE.",
                 fontsize=11, y=1.00)
    plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()
except ImportError:
    print('matplotlib not installed; skipping scatter grid')

## Notes / tuning

- **Real positives**: swap KMeans labels for production descriptors, or pass
  `pos_masks={'balance': BxB bool, ...}` to use a descriptor-similarity graph.
  With no labels for a semantic, that chunk falls back to self-supervised
  (only the two augmented views are positives) and still gets decorrelated.
- **`decorr_weight`**: raise it if the cross-correlation off-diagonal stays
  high (chunks redundant); lower it if specialization (diagonal kNN) suffers
  because the constraint is too aggressive.
- **`var_weight`**: raise if a chunk's per-dim std drifts toward 0.
- **Chunk sizing**: chunks need not be equal — give noisier semantics fewer
  dims. Keep slices disjoint and covering `[0, d_model)`.
- **Scale-out**: wire `labels` through the collator exactly as here and reuse
  `scripts/train_ddp.py`; for very wide batches gather features across ranks
  (mirror VICReg's `_maybe_all_gather`).